# Local Qdrant and embeddings

This optional lab moves from inspectable lexical retrieval to a local vector database. We will preserve payload metadata so authorization and citations remain application responsibilities.

## Architecture

```mermaid
flowchart LR
 D[Documents] --> E[Sentence Transformer]
 E --> Q[Qdrant collection + payload]
 X[Query] --> E --> S[Filtered vector search]
 S --> C[Context + source metadata]
```

Start Qdrant with `docker compose up -d qdrant` and install the optional dependencies from the lesson README.

In [ ]:
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer
from examples.intermediate.qdrant_local import index_documents, search

client = QdrantClient(url='http://localhost:6333')
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
docs = [{'id': 1, 'text': 'Rotate an API key by deploying a replacement.', 'source': 'keys.md', 'metadata': {'tenant_id': 'acme'}}]
index_documents(client, 'rag-learning', docs, encoder, encoder.get_sentence_embedding_dimension())

In [ ]:
hits = search(client, 'How do I rotate a key?', encoder, tenant_id='acme')
[(hit.payload['source'], hit.score) for hit in hits]

## Exercise

Add a second tenant and prove that the payload filter excludes it. Compare vector results with BM25 and record recall, latency, index size, and embedding-model trade-offs.